In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 28. データ テキスト (Data Parallel)

> **学習目標**
> - データ テキスト テキスト テキスト(All-Reduce)テキスト テキスト
> - テキスト テキスト テキスト テキスト
> - DDP (DistributedDataParallel)テキスト テキスト テキスト

## 28.1 テキスト GPUテキスト テキスト

LLM 学習テキスト テキスト·テキスト テキスト:
- 7B モデル FP16 + AdamW = ~70GB
- テキスト テキスト = テキスト テキスト テキスト
- 速度: 1 GPUテキスト テキスト

テキスト: テキスト GPUテキスト テキスト.

## 28.2 データ テキスト テキスト テキスト

1. テキスト モデル テキスト $N$テキスト GPUテキスト テキスト
2. データテキスト $N$テキスト テキスト GPUテキスト テキスト
3. テキスト GPUテキスト テキスト データテキスト テキスト 計算
4. テキスト テキスト テキスト → テキスト GPUテキスト テキスト
5. テキスト GPUテキスト テキスト テキスト テキスト → モデル テキスト テキスト

テキスト:
- テキスト テキスト: $\mathbf{g}_i = \nabla \mathcal{L}_i(\theta)$
- テキスト: $\mathbf{g} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{g}_i$
- テキスト: $\theta \leftarrow \theta - \eta \mathbf{g}$

テキスト テキスト テキスト: $B_{\text{eff}} = N \cdot B_{\text{local}}$


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# データ テキスト テキスト (テキスト GPUテキスト gradient accumulationテキスト テキスト)
class DataParallelSimulator:
    """テキスト GPUテキスト N-GPU DDPテキスト テキスト テキスト."""
    def __init__(self, model, n_gpus=4):
        self.model = model
        self.n_gpus = n_gpus
        # Nテキスト "テキスト GPU" = テキスト Modelテキスト テキスト (gradient accumulationテキスト テキスト)
        # テキスト テキスト Modelテキスト gradientテキスト n_gpusテキスト テキスト
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    def step(self, x_batch, y_batch):
        """x_batch: (N*B, ...), Nテキスト GPU テキスト Bテキスト Sample."""
        # 1. Batchテキスト Nテキスト
        chunks = torch.chunk(x_batch, self.n_gpus)
        y_chunks = torch.chunk(y_batch, self.n_gpus)

        # 2. テキスト "GPU"テキスト Gradient Computation (gradient accumulation)
        self.optimizer.zero_grad()
        total_loss = 0
        for i in range(self.n_gpus):
            out = self.model(chunks[i])
            loss = F.mse_loss(out, y_chunks[i]) / self.n_gpus  # Mean
            loss.backward()  # Gradient テキスト
            total_loss += loss.item() * self.n_gpus

        # 3. optimizer.step()テキスト テキスト Gradientテキスト (テキスト Meanテキスト)
        self.optimizer.step()
        return total_loss / self.n_gpus

# テキスト Modelテキスト テキスト
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 20)
)

dp = DataParallelSimulator(model, n_gpus=4)

# Comparison: テキスト Batch vs DDP テキスト
x = torch.randn(64, 20)
y = torch.randn(64, 20)

loss = dp.step(x, y)
print(f"DDP テキスト 1 step loss: {loss:.4f}")
print(f"n_gpus=4, batch=64 → テキスト GPU batch=16")
print(f"Effectテキスト Batch Magnitude: {dp.n_gpus * 16} = 64")


## 28.3 All-Reduce テキスト

テキスト GPUテキスト テキスト テキスト テキスト GPUテキスト テキスト テキスト テキスト:
- **Ring All-Reduce**: $2(N-1)$ テキスト, テキスト テキスト $|\theta|/N$ テキスト
- **Tree All-Reduce**: テキスト テキスト

テキスト: $O(|\theta|)$ per step. 7B モデルテキスト テキスト テキスト 14GB テキスト.

テキスト テキスト テキスト → テキスト-テキスト テキスト テキスト.


In [ ]:
# All-Reduce テキスト
def all_reduce_sum(grads_list):
    """テキスト GPUテキスト Gradientテキスト Sumテキスト テキスト GPUテキスト テキスト."""
    # Sumテキスト
    total = sum(grads_list)
    # テキスト GPUテキスト テキスト Value テキスト
    return [total.clone() for _ in grads_list]

# 4テキスト GPU テキスト
torch.manual_seed(0)
n_gpus = 4
local_grads = [torch.randn(5) for _ in range(n_gpus)]
print("テキスト GPU テキスト テキスト:")
for i, g in enumerate(local_grads):
    print(f"  GPU {i}: {g.numpy().round(3)}")

# All-Reduce (Mean)
reduced = all_reduce_sum(local_grads)
avg = [r / n_gpus for r in reduced]
print("\nAll-Reduce テキスト (テキスト):")
for i, g in enumerate(avg):
    print(f"  GPU {i}: {g.numpy().round(3)}")
print("\n=> テキスト GPUテキスト テキスト Mean Gradientテキスト テキスト.")


## 28.4 PyTorch DDP (テキスト)

テキスト PyTorch テキスト:
```python
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group('nccl', rank=rank, world_size=world_size)
model = model.to(rank)
model = DDP(model, device_ids=[rank])
# テキスト model(x)テキスト テキスト テキスト テキスト
```

Colab テキスト GPUテキスト テキスト テキスト GPU DDPテキスト テキスト テキスト テキスト, gradient accumulationテキスト テキスト テキスト テキスト テキスト.


In [ ]:
# テキスト GPU DDP テキスト
import time
from llm_math.bench import time_fn

# テキスト モデルテキスト DDP vs テキスト テキスト 比較
def make_model(d=512):
    return nn.Sequential(
        nn.Linear(d, d * 2), nn.ReLU(),
        nn.Linear(d * 2, d), nn.ReLU(),
        nn.Linear(d, d)
    )

# テキスト テキスト テキスト
torch.manual_seed(0)
model_single = make_model()
opt_single = torch.optim.AdamW(model_single.parameters(), lr=1e-3)

def step_single(x, y):
    opt_single.zero_grad()
    out = model_single(x)
    loss = F.mse_loss(out, y)
    loss.backward()
    opt_single.step()
    return loss

# DDP テキスト (4 GPU, テキスト 1/4 テキスト)
torch.manual_seed(0)
model_ddp = make_model()
opt_ddp = torch.optim.AdamW(model_ddp.parameters(), lr=1e-3)

def step_ddp(x, y, n_gpus=4):
    opt_ddp.zero_grad()
    x_chunks = torch.chunk(x, n_gpus)
    y_chunks = torch.chunk(y, n_gpus)
    total_loss = 0
    for xc, yc in zip(x_chunks, y_chunks):
        out = model_ddp(xc)
        loss = F.mse_loss(out, yc) / n_gpus  # テキスト
        loss.backward()  # テキスト テキスト
        total_loss += loss.item() * n_gpus
    opt_ddp.step()
    return total_loss / n_gpus

# 時間 比較 (テキスト テキスト テキスト テキスト)
d = 512
batch_size = 128
x = torch.randn(batch_size, d)
y = torch.randn(batch_size, d)

t_single = time_fn(step_single, x, y, device='cpu', warmup=2, repeat=5)
t_ddp = time_fn(step_ddp, x, y, device='cpu', warmup=2, repeat=5)
print(f"テキスト テキスト Batch (B=128): {t_single['mean_ms']:.3f} ms")
print(f"DDP テキスト (4 GPU, テキスト B=32): {t_ddp['mean_ms']:.3f} ms")
print("\n=> テキスト GPUテキスト DDP テキスト テキスト テキスト (sequential テキスト).")
print("   テキスト テキスト GPUテキスト テキスト テキスト テキスト テキスト.")


## 28.5 Large Batch Training — LARS, LAMB

テキスト テキスト 学習 テキスト. テキスト 学習テキスト テキスト テキスト:

**LARS** (Layer-wise Adaptive Rate Scaling):
$$\eta_\ell = \eta \cdot \frac{\|\theta_\ell\|}{\|\nabla \mathcal{L}_\ell\| + \lambda \|\theta_\ell\|}$$

**LAMB**: LARS + Adam. LLM テキスト 学習テキスト テキスト.

## 28.6 テキスト-テキスト テキスト

バックプロパゲーションテキスト テキスト テキスト. テキスト テキスト テキスト 計算テキスト **テキスト All-Reduce テキスト**, テキスト テキスト バックプロパゲーションテキスト テキスト.

テキスト テキスト 時間テキスト テキスト テキスト テキスト テキスト.

## 28.7 要点

| テキスト | テキスト |
|---|---|
| データ テキスト | テキスト モデル Nテキスト, データ テキスト |
| All-Reduce | テキスト テキスト |
| テキスト テキスト | $N \cdot B_{\text{local}}$ |
| テキスト | $O(\|\theta\|)$ per step |
| LARS/LAMB | テキスト テキスト 学習 テキスト |

## 演習問題

1. 4-GPU DDP テキスト テキスト GPUテキスト テキスト データテキスト 学習テキスト テキスト テキスト テキスト テキスト.
2. テキスト テキスト テキスト 32, 64, 128, 256テキスト 学習テキスト テキスト 比較テキスト.
3. All-Reduceテキスト テキスト $O(\|\theta\|)$テキスト テキスト テキスト.
4. テキスト-テキスト テキスト テキスト 速度テキスト テキスト テキスト.
5. LARSテキスト テキスト テキスト テキスト 度テキスト テキスト テキスト.

> 解答: `solutions/ch28_solutions.ipynb`
